# SE374.R11 Bonus Extension: WSI Low-Mag to High-Mag End-to-End Clinical Pipeline

> **Dự án:** Medical WSSS - Weakly-Supervised Semantic Segmentation on Histopathology Whole Slide Images (WSI)  
> **Công trình tham chiếu mở rộng:** *LowMagWSS (arXiv:2607.10783, 2024/2026)*  
> **Mục tiêu mở rộng (Bonus Feature):** Hiện thực hóa trọn vẹn quy trình phân vùng tự động từ góc nhìn vĩ mô độ phóng đại thấp (Low-Mag: $5\times / 10\times$) đến vi mô độ phóng đại cao (High-Mag: $40\times$) trên toàn bộ tiêu bản bệnh nhân (WSI Gigapixel), tiết kiệm $80\% - 92\%$ tài nguyên tính toán GPU.

## 1. Cài đặt môi trường & Kết nối Google Drive

In [ ]:
# Mount Google Drive
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

# Ensure project root is in sys.path
PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

!pip install -q gdown albumentations tifffile
print("[OK] Môi trường và đường dẫn sẵn sàng:", PROJECT_ROOT)

## 2. Nạp Thư Viện & Checkpoints Mô Hình Pretrained

In [ ]:
import torch
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

from src.colab_pkg.downloader import ColabArtifactDownloader
from src.colab_pkg.model_loader import ColabModelLoader
from src.models.wsss_inference_methods import MultiMethodWSSSEngineV2
from src.pipeline.wsi_lowmag_highmag_pipeline import WSILowMagHighMagPipeline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[OK] Sử dụng thiết bị tính toán:', device)

# Tự động tải 2 checkpoint chính thức nếu chưa có trên ổ đĩa
downloader = ColabArtifactDownloader()
ckpt_stage1_path, ckpt_stage2_path = downloader.download_checkpoints()

# Nạp mô hình Stage-1 Net_CAM và Stage-2 DeepLabV3+
loader = ColabModelLoader(device=device)
model_stage1 = loader.load_stage1_netcam(ckpt_stage1_path)
model_stage2 = loader.load_stage2_deeplabv3plus(ckpt_stage2_path)
print('[OK] Đã nạp thành công 2 mô hình Stage-1 (Net_CAM) và Stage-2 (DeepLabV3+)!')

## 3. Khởi Tạo Pipeline WSI Low-Mag to High-Mag & Nạp Tiêu Bản Bệnh Nhân

In [ ]:
pipeline = WSILowMagHighMagPipeline(
    patch_size=224,
    lowmag_target_size=(1024, 1024),
    tissue_threshold=0.15,
    device=device
)

# Sử dụng ảnh tiêu bản mô bệnh học giải phẫu bệnh đại diện
sample_slide_path = 'product/proposal/assets/fig1_standard_wsi_pipeline_scientific.png'
if not os.path.exists(sample_slide_path):
    sample_slide_path = 'product/proposal/assets/fig1_standard_wsi_pipeline.png'

overview_img, orig_dims = pipeline.read_slide_overview(sample_slide_path)
print(f'[OK] Nạp tiêu bản WSI thành công:')
print(f' - Kích thước gốc toàn slide: {orig_dims[0]:,} x {orig_dims[1]:,} pixels')
print(f' - Kích thước tầng Low-Mag (5x/10x): {overview_img.size[0]} x {overview_img.size[1]} pixels')

## 4. Phase 1: Quét Nhanh Góc Rộng Ở Low-Mag (5x) & Bản Đồ Nhiệt Vĩ Mô

In [ ]:
import time
t_start = time.time()
tissue_mask, macro_probs = pipeline.compute_macro_tissue_heatmap(overview_img, model_stage1)
scan_sec = time.time() - t_start

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(overview_img)
axes[0].set_title('1. Tiêu bản Macro gốc (Low-Mag 5x)', fontsize=13, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(tissue_mask, cmap='gray')
axes[1].set_title('2. Mặt nạ tách mô Otsu (Loại bỏ kính trắng)', fontsize=13, fontweight='bold')
axes[1].axis('off')

im = axes[2].imshow(macro_probs[0], cmap='jet')
axes[2].set_title('3. Bản đồ nhiệt vùng nghi ngờ khối u (Macro CAM)', fontsize=13, fontweight='bold')
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

print(f'[THÀNH CÔNG] Thời gian quét toàn bộ slide ở Low-Mag: {scan_sec:.3f} giây!')

## 5. Phase 2: Đánh Giá Tỷ Lệ Cắt Giảm Tính Toán Nhờ Định Hướng Low-Mag

In [ ]:
candidates, total_possible, reduction_pct = pipeline.select_candidate_highmag_patches(
    orig_dims, tissue_mask, macro_probs
)

print('='*75)
print('BÁO CÁO HIỆU NĂNG TIẾT KIỆM TÀI NGUYÊN (LOW-MAG GUIDED REDUCTION)')
print('='*75)
print(f' - Tổng số patch lý thuyết nếu quét mù quáng cả slide 40x: {total_possible:,} patches')
print(f' - Số patch nghi ngờ được Low-Mag chọn lọc để zoom 40x:   {len(candidates):,} patches')
print(f' - TỶ LỆ CẮT GIẢM TÍNH TOÁN & THỜI GIAN GPU ĐẠT ĐƯỢC:      {reduction_pct:.2f}%')
print('='*75)

## 6. Phase 3 & 4: Zoom Cực Đại 40x (M6 SOTA) & Ghép Mặt Nạ Toàn Tiêu Bản

In [ ]:
# Chạy suy luận chi tiết trên các patch ứng viên và ghép mặt nạ
stitched_mask, report = pipeline.run_end_to_end_wsi(
    slide_input=overview_img,
    model_stage2=model_stage2,
    model_stage1=model_stage1,
    method='M6_MultiScale_Pyramid_D4_Ensemble',
    slide_id='PATIENT_WSI_DEMO_01',
    max_patches_to_infer=15  # Demo nhanh trên 15 patch đại diện
)

color_stitched = pipeline.colorize_mask(stitched_mask)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(overview_img)
axes[0].set_title('Tiêu bản gốc của bệnh nhân (Macro 5x)', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(color_stitched)
axes[1].set_title('Mặt nạ phân vùng tái cấu trúc toàn tiêu bản (High-Mag Stitched Mask)', fontsize=14, fontweight='bold')
axes[1].axis('off')
plt.tight_layout()
plt.show()

print(f'[THÀNH CÔNG] Phân tích hoàn tất tiêu bản bệnh nhân:')
print(f' - Tỷ lệ diện tích khối u xâm lấn: {report.tumor_fraction_pct}%')
print(f' - Tổng thời gian quét & phân đoạn: {report.total_elapsed_time_sec} giây')

## 7. Bảng Tóm Tắt Chẩn Đoán Lâm Sàng Cho Bác Sĩ (Clinical Summary)

In [ ]:
print('='*70)
print(f'PHIẾU KẾT QUẢ PHÂN ĐOẠN GIẢI PHẪU BỆNH SỐ (WSI AI REPORT): {report.slide_id}')
print('='*70)
print(f'Phương pháp suy luận:          {report.selected_method}')
print(f'Độ phân giải phân tích:        {report.macro_dimensions[0]}x{report.macro_dimensions[1]} pixels')
print(f'Diện tích mô sống phát hiện:   {report.tissue_area_px:,} pixels')
print(f'Diện tích Khối u (Tumor):      {report.tumor_area_px:,} pixels ({report.tumor_fraction_pct}%)')
print(f'Diện tích Mô đệm (Stroma):     {report.stroma_area_px:,} pixels')
print(f'Diện tích Bạch cầu (Lympho):   {report.lymphocyte_area_px:,} pixels')
print(f'Diện tích Hoại tử (Necrosis):  {report.necrosis_area_px:,} pixels')
print(f'Thời gian quét Low-Mag:        {report.lowmag_scan_time_sec} giây')
print(f'Thời gian suy luận High-Mag:   {report.highmag_inference_time_sec} giây')
print(f'Hiệu quả giảm tải tính toán:   TIẾT KIỆM {report.compute_reduction_pct}%')
print('='*70)